In [25]:
import pandas as pd
import numpy as np

raw_data = {
    'record_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'age': [25, -3, 28.2, 150, 32, np.nan], 
    'join_date': ['2023-01-15', '2023/02/28', '15-03-2023', 'Not a Date', '2023-05-10', '2023-06-01'], 
    'score': ['88.5', 92.0, np.nan, 'Error_Code_X', 75.5, 80.0] 
}

df = pd.DataFrame(raw_data)

In [26]:
# 查看数据信息
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   record_id  6 non-null      int64  
 1   age        5 non-null      float64
 2   join_date  6 non-null      object 
 3   score      5 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 324.0+ bytes
None


In [27]:
# record_id列类型显示为int64,ID没有实际的数学意义，转为str
data_clean = df.copy()
data_clean['record_id'] = data_clean['record_id'].astype(str)
print(data_clean['record_id'].dtype)

object


In [28]:
# 年龄应该是整数类型
# 查看年龄列的数值范围
print(data_clean['age'].describe())

# 去除age列的异常值
# 抓取无效年龄值（布尔序列）
invalid_age_mask = (data_clean['age'] < 0) | (data_clean['age'] > 120)
# print(invalid_age_mask)
# print(data_clean[invalid_age_mask])
# 将无效年龄替换为NaN
data_clean.loc[invalid_age_mask,'age'] = np.nan
# print(data_clean)

# 将age列的浮点数向下取整,并转换为整数类型
data_clean['age'] = np.floor(data_clean['age']).astype('Int8')
print(f"查看清洗后的age列\n{data_clean['age']}")
print(data_clean['age'].dtype)



count      5.000000
mean      46.440000
std       59.518636
min       -3.000000
25%       25.000000
50%       28.200000
75%       32.000000
max      150.000000
Name: age, dtype: float64
查看清洗后的age列
0      25
1    <NA>
2      28
3    <NA>
4      32
5    <NA>
Name: age, dtype: Int8
Int8


In [29]:
# 处理join_date

# 由于不知道join_data的脏数据类型，因此采取“压力测试”
parsed_dates = pd.to_datetime(data_clean['join_date'],errors='coerce',format='mixed')
print(parsed_dates)

# 找出原本不是空值，被解析后变为空值的数据（即脏数据类型）
mask_bad_dates = (data_clean['join_date'].notna()) & (parsed_dates.isna())

bad_date_examples = data_clean.loc[mask_bad_dates,'join_date']
print(bad_date_examples)

# 如果脏数据类型数量过大，不可能全部打印出来。
# 不需要看清每一个长什么样，只需要看哪种脏类型出现得最频繁
dirty_stype_rank = bad_date_examples.value_counts().head()
print(f"脏数据排行榜：")
print(dirty_stype_rank)

# 如果脏数据类型非常分散，几万条脏数据类型都不一样，就要把具体得文本抽象化，统计脏数据得长度分布
# 正常日期长度一般为10
length_distribution = bad_date_examples.str.len().value_counts()
print(length_distribution)

# 将data_clean得join_date替换为parsed_dates
data_clean['join_date'] = parsed_dates
print(data_clean['join_date'])

0   2023-01-15
1   2023-02-28
2   2023-03-15
3          NaT
4   2023-05-10
5   2023-06-01
Name: join_date, dtype: datetime64[ns]
3    Not a Date
Name: join_date, dtype: object
脏数据排行榜：
join_date
Not a Date    1
Name: count, dtype: int64
join_date
10    1
Name: count, dtype: int64
0   2023-01-15
1   2023-02-28
2   2023-03-15
3          NaT
4   2023-05-10
5   2023-06-01
Name: join_date, dtype: datetime64[ns]


In [30]:
# 清洗score

# 上大重量做压力测试，将score强转为数值，遇到无法解析得乱码，强制转为NaN
parsed_scores = pd.to_numeric(data_clean['score'],errors='coerce')
print(parsed_scores)

# 找出转换前不为空值，但转换后为空值的数据
mask_bad_score = (data_clean['score'].notna()) & (parsed_scores.isna())
bad_score_examples = data_clean.loc[mask_bad_score,'score']
print(bad_score_examples)

# 统计脏数据类型数量排行
score_dirty_stype_rank = bad_score_examples.value_counts().head()
print(score_dirty_stype_rank)

# 将清洗后的数据存进data_clean
data_clean['score'] = parsed_scores
print(data_clean)
print(data_clean.info())

0    88.5
1    92.0
2     NaN
3     NaN
4    75.5
5    80.0
Name: score, dtype: float64
3    Error_Code_X
Name: score, dtype: object
score
Error_Code_X    1
Name: count, dtype: int64
  record_id   age  join_date  score
0      1001    25 2023-01-15   88.5
1      1002  <NA> 2023-02-28   92.0
2      1003    28 2023-03-15    NaN
3      1004  <NA>        NaT    NaN
4      1005    32 2023-05-10   75.5
5      1006  <NA> 2023-06-01   80.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   record_id  6 non-null      object        
 1   age        3 non-null      Int8          
 2   join_date  5 non-null      datetime64[ns]
 3   score      4 non-null      float64       
dtypes: Int8(1), datetime64[ns](1), float64(1), object(1)
memory usage: 288.0+ bytes
None
